# 集群上lumpy运行流程

## 一、数据来源

### 197个样本路径：/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/

### 80组配对样本名称列表：/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt

## 二、配置环境与运行 

In [2]:
conda create -n lumpy python=2.7 -y
conda activate lumpy

conda install -c bioconda lumpy-sv svtyper samtools bcftools samblaster -y

conda install -c conda-forge openssl=1.0 -y

# 输入下面命令看是否成功安装
lumpyexpress -h
svtyper -h

SyntaxError: invalid syntax (2371128687.py, line 1)

### 1、单组配对样本测试脚本/mnt/home/ygjx/chenkejin/Lumpy/test_single_lumpy.sh，代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=lumpy_1866277           # 作业名称
#SBATCH --nodes=1                          # 申请 1 个节点
#SBATCH --cpus-per-task=12                 # 为该任务分配 12 个 CPU
#SBATCH --mem=48G                          # [新增] 申请充足内存，防止 OOM 被杀
#SBATCH --output=/mnt/home/ygjx/chenkejin/Lumpy/lumpy_1866277_%j.log   # 标准输出和报错合并写入此日志

set -euo pipefail

# --- 0. 激活环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate lumpy_env

# --- 1. 软件环境配置 ---
SAMTOOLS=samtools
SVTYPER=svtyper
BCFTOOLS=bcftools
EXTRACT_SCRIPT="/mnt/home/ygjx/chenkejin/anaconda3/envs/lumpy_env/share/lumpy-sv-0.2.13-0/scripts/extractSplitReads_BwaMem"
THREADS=24

# --- 2. 路径与样本配置 ---
SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/Lumpy"

NORMAL_ID="1866277N"
TUMOR_ID="1866277T"
PREFIX="1866277" 

echo "=========================================================="
echo "[$(date)] 开始处理测试样本对: ${PREFIX}"
echo "分配的 CPU 核心数: ${THREADS}"
echo "=========================================================="

# --- 3. 建立工作目录结构 ---
INPUT_DIR="${WORK_DIR}/inputs/${PREFIX}"
SPLITTER_DIR="${WORK_DIR}/splitters/${PREFIX}"
VCF_DIR="${WORK_DIR}/VCF/${PREFIX}"
TMP_DIR="${WORK_DIR}/tmp/${PREFIX}"

mkdir -p "$INPUT_DIR" "$SPLITTER_DIR" "$VCF_DIR" "$TMP_DIR"

# --- 4. 自动检索真实文件并创建软链接 ---
echo "[$(date)] 正在定位 BAM 文件并创建软链接..."

# [修正] 指定最终版的 BAM 后缀，绝对防止抓错中间文件
NORMAL_BAM_SRC=$(ls ${SOURCE_BASE}/${NORMAL_ID}/*.sorted.markdup.BQSR.bam | head -n 1)
NORMAL_BAI_SRC=$(ls ${SOURCE_BASE}/${NORMAL_ID}/*.bai | head -n 1)
TUMOR_BAM_SRC=$(ls ${SOURCE_BASE}/${TUMOR_ID}/*.sorted.markdup.BQSR.bam | head -n 1)
TUMOR_BAI_SRC=$(ls ${SOURCE_BASE}/${TUMOR_ID}/*.bai | head -n 1)

NORMAL_BAM="${INPUT_DIR}/$(basename ${NORMAL_BAM_SRC})"
TUMOR_BAM="${INPUT_DIR}/$(basename ${TUMOR_BAM_SRC})"

ln -sf "${NORMAL_BAM_SRC}" "${NORMAL_BAM}"
ln -sf "${NORMAL_BAI_SRC}" "${NORMAL_BAM}.bai" 
ln -sf "${NORMAL_BAI_SRC}" "${INPUT_DIR}/$(basename ${NORMAL_BAI_SRC})"

ln -sf "${TUMOR_BAM_SRC}" "${TUMOR_BAM}"
ln -sf "${TUMOR_BAI_SRC}" "${TUMOR_BAM}.bai"
ln -sf "${TUMOR_BAI_SRC}" "${INPUT_DIR}/$(basename ${TUMOR_BAI_SRC})"

echo "  Tumor 软链接:  $TUMOR_BAM"
echo "  Normal 软链接: $NORMAL_BAM"

# --- 5. 定义中间和输出文件 ---
tumor_splitters_bam="${SPLITTER_DIR}/${PREFIX}.tumor.splitters.bam"
normal_splitters_bam="${SPLITTER_DIR}/${PREFIX}.normal.splitters.bam"
tumor_discordants_bam="${SPLITTER_DIR}/${PREFIX}.tumor.discordants.bam"
normal_discordants_bam="${SPLITTER_DIR}/${PREFIX}.normal.discordants.bam"

tumor_normal_vcf="${WORK_DIR}/${PREFIX}.tumor_normal.lumpy.vcf"
lumpy_genotyped_vcf="${WORK_DIR}/${PREFIX}.lumpy.genotyped.vcf"
vcf_output="${VCF_DIR}/${PREFIX}.lumpy.somatic.vcf"
vcf_filter_output="${VCF_DIR}/${PREFIX}.lumpy.somatic.filtered.vcf"


# --- 6. LUMPY 核心计算流程 ---

# [核心逻辑修改]：定义标准染色体白名单
STD_CHRS="chr1 chr2 chr3 chr4 chr5 chr6 chr7 chr8 chr9 chr10 chr11 chr12 chr13 chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 chr22 chrX chrY"

# Step 1: Extract Splitters (源头拦截：只提取 STD_CHRS 包含的染色体)
echo "[$(date)] Step 1: 提取 Splitters (仅限标准染色体)..."
$SAMTOOLS view -@ $THREADS -h "$TUMOR_BAM" $STD_CHRS | "$EXTRACT_SCRIPT" -i stdin | $SAMTOOLS view -@ $THREADS -Sb - > "$tumor_splitters_bam"
$SAMTOOLS view -@ $THREADS -h "$NORMAL_BAM" $STD_CHRS | "$EXTRACT_SCRIPT" -i stdin | $SAMTOOLS view -@ $THREADS -Sb - > "$normal_splitters_bam"

# Step 2: Extract Discordants (源头拦截：只提取 STD_CHRS 包含的染色体)
echo "[$(date)] Step 2: 提取 Discordants (仅限标准染色体)..."
$SAMTOOLS view -@ $THREADS -b -F 1294 "$TUMOR_BAM" $STD_CHRS > "$tumor_discordants_bam"
$SAMTOOLS view -@ $THREADS -b -F 1294 "$NORMAL_BAM" $STD_CHRS > "$normal_discordants_bam"

# Step 3: Lumpy Express 
echo "[$(date)] Step 3: 运行 LumpyExpress..."
lumpyexpress \
    -B "$TUMOR_BAM","$NORMAL_BAM" \
    -S "$tumor_splitters_bam","$normal_splitters_bam" \
    -D "$tumor_discordants_bam","$normal_discordants_bam" \
    -o "$tumor_normal_vcf" \
    -T "$TMP_DIR"

# Step 4: SVTyper Genotyping
echo "[$(date)] Step 4: 运行 SVTyper 基因分型..."
$SVTYPER \
    -B "$TUMOR_BAM","$NORMAL_BAM" \
    -i "$tumor_normal_vcf" \
    -o "$lumpy_genotyped_vcf"

# Step 5: Somatic Filter
echo "[$(date)] Step 5: 运行 Somatic 变异过滤..."
$BCFTOOLS view \
    -i 'GT[0]="het" && GT[1]="RR"' \
    "$lumpy_genotyped_vcf" > "$vcf_output"

# Step 6: 染色体白名单清理
echo "[$(date)] Step 6: 染色体白名单清理..."
# 此时传进来的数据已经极度干净，这一步作为最后一道保险瞬间完成
awk '/^#/ || $1 ~ /^chr[0-9XY]+$/' \
    "$vcf_output" \
    > "$vcf_filter_output"

# --- 7. 清理垃圾 ---
echo "[$(date)] Step 7: 清理特征提取产生的巨大中间 BAM 文件..."
rm -f "$tumor_splitters_bam" "$normal_splitters_bam" "$tumor_discordants_bam" "$normal_discordants_bam"

echo "=========================================================="
echo "[$(date)] 测试样本对 ${PREFIX} 运行完毕！"
echo "最终结果保存在: $vcf_filter_output"
echo "=========================================================="

### sbatch /mnt/home/ygjx/chenkejin/Lumpy/test_single_lumpy.sh 提交作业

### 2、集群批量处理脚本/mnt/home/ygjx/chenkejin/Lumpy/batch_lumpy.sh，代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=Lumpy_Batch             # 作业名称
#SBATCH --nodes=1                          # 每个子任务申请 1 个节点
#SBATCH --cpus-per-task=8                  # 每个子任务分配 8 个 CPU
#SBATCH --mem=48G                          # 防止 Lumpy 内存溢出
#SBATCH --array=1-80%15                    # 共 80 对样本，每次最多同时运行 15 个
#SBATCH --output=/mnt/home/ygjx/chenkejin/Lumpy/logs/slurm_array_%A_%a.out # 占位日志，实际内容会动态重定向

set -euo pipefail

# --- 0. 激活环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate lumpy_env

# --- 1. 全局变量与路径 ---
SAMTOOLS=samtools
SVTYPER=svtyper
BCFTOOLS=bcftools
EXTRACT_SCRIPT="/mnt/home/ygjx/chenkejin/anaconda3/envs/lumpy_env/share/lumpy-sv-0.2.13-0/scripts/extractSplitReads_BwaMem"
THREADS=8

SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/Lumpy"
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"

MASTER_LOG="${WORK_DIR}/master_progress.log"

# --- 2. 任务解析 ---
LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$TASK_LIST")
NORMAL_ID=$(echo "$LINE" | awk '{print $1}')
TUMOR_ID=$(echo "$LINE" | awk '{print $2}')
PREFIX=$(echo "$NORMAL_ID" | sed 's/N//')

# 【防崩溃保险】如果读到空行，直接安全退出
if [ -z "$PREFIX" ]; then
    exit 0
fi

# --- 3. 专属日志动态重定向 ---
mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.log"
# 从这一行开始，所有的终端输出都会被实时写入到以样本命名的 .log 文件中
exec > >(tee -i "$SAMPLE_LOG") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] 样本 ${PREFIX} 开始处理"
echo "任务阵列 ID: ${SLURM_ARRAY_TASK_ID}/80  执行节点: $(hostname)"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX}" >> "$MASTER_LOG"

# --- 4. 建立专属小沙盒 ---
SANDBOX="${WORK_DIR}/sandbox/${PREFIX}"
INPUT_DIR="${SANDBOX}/inputs"
SPLITTER_DIR="${SANDBOX}/splitters"
TMP_DIR="${SANDBOX}/tmp"
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"

# 创建该样本专属的隔离文件夹
mkdir -p "$INPUT_DIR" "$SPLITTER_DIR" "$TMP_DIR" "$FINAL_VCF_DIR"

# --- 5. 精准拉取数据文件 ---
# 根据你提供的命名规则，直接拼接绝对路径，彻底避免抓错文件
NORMAL_BAM_SRC="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM_SRC="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

# 索引文件可能是 .bai 或 .bam.bai，用 ls 抓取最稳妥
NORMAL_BAI_SRC=$(ls ${SOURCE_BASE}/${NORMAL_ID}/*.bai | head -n 1)
TUMOR_BAI_SRC=$(ls ${SOURCE_BASE}/${TUMOR_ID}/*.bai | head -n 1)

NORMAL_BAM="${INPUT_DIR}/$(basename ${NORMAL_BAM_SRC})"
TUMOR_BAM="${INPUT_DIR}/$(basename ${TUMOR_BAM_SRC})"

# 创建沙盒内的软链接
ln -sf "${NORMAL_BAM_SRC}" "${NORMAL_BAM}"
ln -sf "${NORMAL_BAI_SRC}" "${NORMAL_BAM}.bai" 
ln -sf "${TUMOR_BAM_SRC}" "${TUMOR_BAM}"
ln -sf "${TUMOR_BAI_SRC}" "${TUMOR_BAM}.bai"

# --- 6. 定义沙盒内的中间文件 ---
tumor_splitters_bam="${SPLITTER_DIR}/${PREFIX}.tumor.splitters.bam"
normal_splitters_bam="${SPLITTER_DIR}/${PREFIX}.normal.splitters.bam"
tumor_discordants_bam="${SPLITTER_DIR}/${PREFIX}.tumor.discordants.bam"
normal_discordants_bam="${SPLITTER_DIR}/${PREFIX}.normal.discordants.bam"

tumor_normal_vcf="${SANDBOX}/${PREFIX}.tumor_normal.lumpy.vcf"
lumpy_genotyped_vcf="${SANDBOX}/${PREFIX}.lumpy.genotyped.vcf"
vcf_output="${SANDBOX}/${PREFIX}.lumpy.somatic.vcf"
vcf_filter_output="${FINAL_VCF_DIR}/${PREFIX}.lumpy.somatic.filtered.vcf"

# 【用完即焚】无论如何都会清理几百GB的拆分文件
trap 'echo "[$(date)] 正在清理专属沙盒的巨大中间 BAM 文件..."; rm -f "$tumor_splitters_bam" "$normal_splitters_bam" "$tumor_discordants_bam" "$normal_discordants_bam"' EXIT INT TERM

# 断点续传检查
if [ -f "$vcf_filter_output" ]; then
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] 样本 ${PREFIX} 已存在结果，跳过运行。"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] Sample ${PREFIX} skipped" >> "$MASTER_LOG"
    exit 0
fi

# --- 7. 核心流程执行 ---
{
    STD_CHRS="chr1 chr2 chr3 chr4 chr5 chr6 chr7 chr8 chr9 chr10 chr11 chr12 chr13 chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 chr22 chrX chrY"

    echo "[$(date)] Step 1: 提取 Splitters (仅限标准染色体)..."
    $SAMTOOLS view -@ $THREADS -h "$TUMOR_BAM" $STD_CHRS | "$EXTRACT_SCRIPT" -i stdin | $SAMTOOLS view -@ $THREADS -Sb - > "$tumor_splitters_bam"
    $SAMTOOLS view -@ $THREADS -h "$NORMAL_BAM" $STD_CHRS | "$EXTRACT_SCRIPT" -i stdin | $SAMTOOLS view -@ $THREADS -Sb - > "$normal_splitters_bam"

    echo "[$(date)] Step 2: 提取 Discordants (仅限标准染色体)..."
    $SAMTOOLS view -@ $THREADS -b -F 1294 "$TUMOR_BAM" $STD_CHRS > "$tumor_discordants_bam"
    $SAMTOOLS view -@ $THREADS -b -F 1294 "$NORMAL_BAM" $STD_CHRS > "$normal_discordants_bam"

    echo "[$(date)] Step 3: 运行 LumpyExpress..."
    lumpyexpress \
        -B "$TUMOR_BAM","$NORMAL_BAM" \
        -S "$tumor_splitters_bam","$normal_splitters_bam" \
        -D "$tumor_discordants_bam","$normal_discordants_bam" \
        -o "$tumor_normal_vcf" \
        -T "$TMP_DIR"

    echo "[$(date)] Step 4: 运行 SVTyper 基因分型..."
    $SVTYPER \
        -B "$TUMOR_BAM","$NORMAL_BAM" \
        -i "$tumor_normal_vcf" \
        -o "$lumpy_genotyped_vcf"

    echo "[$(date)] Step 5: 运行 Somatic 变异过滤..."
    $BCFTOOLS view \
        -i 'GT[0]="het" && GT[1]="RR"' \
        "$lumpy_genotyped_vcf" > "$vcf_output"

    echo "[$(date)] Step 6: 染色体白名单清理..."
    awk '/^#/ || $1 ~ /^chr[0-9XY]+$/' "$vcf_output" > "$vcf_filter_output"

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 样本 ${PREFIX} 运行成功！"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} completed" >> "$MASTER_LOG"

} || {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] 样本 ${PREFIX} 运行失败！"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} failed!" >> "$MASTER_LOG"
    exit 1
}

### sbatch /mnt/home/ygjx/chenkejin/Lumpy/batch_lumpy.sh 提交作业，tail -f [日志文件]  可实时查看日志

## 结果文件路径：/mnt/home/ygjx/chenkejin/Lumpy/Final_Results/